In [3]:
import pandas as pd 
import numpy as np 
df = pd.read_csv("../data/cleaned/paimana_cleaned.csv" )
print(df.shape)

(7594, 19)


In [7]:
# Feature 1 — Cost Overrun %
df["COST_OVERRUN_PCT"] = np.where(    
    df["ORIGINAL_COST_RS_CRORE"] > 0, ((
            df["REVISED_COST_RS_CRORE"] - df["ORIGINAL_COST_RS_CRORE"]        
        ) /               
    df["ORIGINAL_COST_RS_CRORE"]) * 100,    
    np.nan 
)

In [8]:
# Feature 2 — Expenditure %
df["EXPENDITURE_PCT"] = np.where(    
    df["ORIGINAL_COST_RS_CRORE"] > 0,(        
        df["CUMULATIVE_EXPENDITURE_RS_CRORE"] / df["ORIGINAL_COST_RS_CRORE"]    ) * 100,   
    np.nan 
)

In [9]:
# Feature 3 — Progress gap
df["PROGRESS_GAP"] = (df["EXPENDITURE_PCT"] - df["PHYSICAL_PROGRESS_PERCENT"])

In [27]:
# Feature 4 — Project duration
df["START_DATE"] = pd.to_datetime(df["START_DATE"], format="mixed")
df["ORIGINAL_TARGET_DOC"] = pd.to_datetime(df["ORIGINAL_TARGET_DOC"], format="mixed")
df["ORIGINAL_DURATION_DAYS"] = (df["ORIGINAL_TARGET_DOC"] - df["START_DATE"]).dt.days

In [29]:
# Feature 5 — Schedule change

df["REVISED_DOC"] = pd.to_datetime(df["REVISED_DOC"], format="mixed")
df["SCHEDULE_CHANGE_DAYS"] = (df["REVISED_DOC"] - df["ORIGINAL_TARGET_DOC"]).dt.days

In [32]:
# Feature 6 — Progress change

df = df.sort_values(["PROJECT_CODES", "REPORT_MONTH"])
df["PROGRESS_CHANGE"] = (    
    df.groupby("PROJECT_CODES")
    ["PHYSICAL_PROGRESS_PERCENT"]   
    .diff() )

In [33]:
# Feature 7 — Expenditure change
df["EXPENDITURE_CHANGE"] = (    
    df.groupby("PROJECT_CODES")      
    ["CUMULATIVE_EXPENDITURE_RS_CRORE"]      
    .diff() )

In [34]:
# Feature 8 — Revised cost change
df["REVISED_COST_CHANGE"] = (    
    df.groupby("PROJECT_CODES")      
    ["REVISED_COST_RS_CRORE"]      
    .diff() )

In [35]:
# Feature 9 — Delay indicator
df["DELAY_INDICATOR"] = np.where( df["SCHEDULE_CHANGE_DAYS"] > 0, 1, 0 )

In [36]:
latest = (    df.sort_values("REPORT_MONTH")      .groupby("PROJECT_CODES")      .tail(1) ) 
print(latest.shape)

(3580, 28)


In [37]:
latest.to_csv(    "../data/processed/project_master.csv",    index=False ) 
print("project_master.csv created")

project_master.csv created


In [38]:
print(df.columns.tolist())

['MINISTRY', 'SECTOR', 'SLNO', 'PROJECT_NAME', 'AGENCY', 'PROJECT_CODES', 'STATE', 'DATE_OF_APPROVAL', 'START_DATE', 'ORIGINAL_TARGET_DOC', 'REVISED_DOC', 'ORIGINAL_COST_RS_CRORE', 'REVISED_COST_RS_CRORE', 'CUMULATIVE_EXPENDITURE_RS_CRORE', 'PHYSICAL_PROGRESS_PCT', 'SOURCE_PAGE', 'REPORT_MONTH', 'SOURCE_FILE', 'PHYSICAL_PROGRESS_PERCENT', 'COST_OVERRUN_PCT', 'EXPENDITURE_PCT', 'PROGRESS_GAP', 'ORIGINAL_DURATION_DAYS', 'SCHEDULE_CHANGE_DAYS', 'PROGRESS_CHANGE', 'EXPENDITURE_CHANGE', 'REVISED_COST_CHANGE', 'DELAY_INDICATOR']


In [43]:
cost_features = [    
    "MINISTRY",    
    "SECTOR",    
    "STATE",    
    "AGENCY",    
    "ORIGINAL_COST_RS_CRORE",    
    "CUMULATIVE_EXPENDITURE_RS_CRORE",    
    "PHYSICAL_PROGRESS_PERCENT",    
    "ORIGINAL_DURATION_DAYS",    
    "PROGRESS_CHANGE",    
    "EXPENDITURE_CHANGE" ]

available_features = [col for col in cost_features 
                      if col in df.columns]
missing_features = [col for col in cost_features 
                    if col not in df.columns]

print("Available:")
print(available_features)
print("\nMissing:")
print(missing_features)


Available:
['MINISTRY', 'SECTOR', 'STATE', 'AGENCY', 'ORIGINAL_COST_RS_CRORE', 'CUMULATIVE_EXPENDITURE_RS_CRORE', 'PHYSICAL_PROGRESS_PERCENT', 'ORIGINAL_DURATION_DAYS', 'PROGRESS_CHANGE', 'EXPENDITURE_CHANGE']

Missing:
[]


In [44]:
cost_dataset = df[    available_features ].copy()
cost_dataset["COST_OVERRUN_PCT"] = df[    "COST_OVERRUN_PCT" ]

In [45]:
cost_dataset.to_csv(    "../data/processed/cost_model_dataset.csv",    index=False )

In [46]:
time_features = [    
    "MINISTRY",    
    "SECTOR",    
    "STATE",    
    "AGENCY",    
    "ORIGINAL_COST_RS_CRORE",    
    "CUMULATIVE_EXPENDITURE_RS_CRORE",    
    "PHYSICAL_PROGRESS_PERCENT",    
    "ORIGINAL_DURATION_DAYS",    
    "PROGRESS_CHANGE",    
    "EXPENDITURE_CHANGE" ] 
available_time_features = [    
    col for col in time_features    
    if col in df.columns ] 
time_dataset = df[    available_time_features ].copy()

In [47]:
time_dataset["DELAY_INDICATOR"] = df[    "DELAY_INDICATOR" ]

In [48]:
time_dataset.to_csv(    "../data/processed/time_model_dataset.csv",    index=False)